In [1]:
#VGG11在cifar10上的实战
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)


sys.version_info(major=3, minor=13, micro=5, releaselevel='final', serial=0)
cuda:0


In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("D:/cifar-10")

train_lables_file = DATA_DIR / "trainLabels.csv"
test_csv_file = DATA_DIR / "sampleSubmission.csv"
train_folder = DATA_DIR / "train/train"
test_folder = DATA_DIR / "test/test"

class_names = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

def load_image_paths_and_labels(labels_csv_path, images_folder):
    df = pd.read_csv(labels_csv_path)
    items = []
    for idx, row in df.iterrows():
        img_name = row['id']
        label = row['label']
        img_path = images_folder / f"{img_name}.png"
        if not img_path.exists():
            img_path = images_folder / f"{img_name}.jpg"
        items.append( (str(img_path), label) )
    return items

train_items = load_image_paths_and_labels(train_lables_file, train_folder)

import pprint
pprint.pprint(train_items[:5])

[('D:\\cifar-10\\train\\train\\1.png', 'frog'),
 ('D:\\cifar-10\\train\\train\\2.png', 'truck'),
 ('D:\\cifar-10\\train\\train\\3.png', 'truck'),
 ('D:\\cifar-10\\train\\train\\4.png', 'deer'),
 ('D:\\cifar-10\\train\\train\\5.png', 'automobile')]


In [3]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

train_items_ = train_items[:45000]
valid_items_ = train_items[45000:]

train_df = pd.DataFrame(train_items_, columns=["img_path", "label"])
valid_df = pd.DataFrame(valid_items_, columns=["img_path", "label"])

class Cifar10Dataset(Dataset):
    df_map = {
        "train": train_df,
        "eval": valid_df,
    }
    label_to_idx = {label: idx for idx, label in enumerate(class_names)}
    idx_to_label = {idx: label for idx, label in enumerate(class_names)}
    def __init__(self, mode, transform=None):
        self.df = self.df_map.get(mode, None)
        if self.df is None:
            raise ValueError("mode should be one of train, val, test, but got {}".format(mode))
        self.transform = transform

    def __getitem__(self, index):
        img_path, label = self.df.iloc[index]
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)
        label = self.label_to_idx[label]
        return img, label

    def __len__(self):
        return self.df.shape[0]

IMAGE_SIZE = 32
mean, std = [0.4914, 0.4822, 0.4465], [0.247, 0.243, 0.261]

transforms_train = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomRotation(40),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

transforms_eval = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
train_ds = Cifar10Dataset("train", transforms_train)
eval_ds = Cifar10Dataset("eval", transforms_eval)

In [4]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"训练集样本数: {len(train_ds)}")
print(f"验证集样本数: {len(eval_ds)}")
print(batch_size)


训练集样本数: 45000
验证集样本数: 5000
32


In [5]:
import torch.nn as nn

class VGG11(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), 
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), 
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1), 
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), 
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),
        )
        
        self._initialize_weights()
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
model = VGG11(num_classes=len(class_names))


In [6]:
import torch

dummy_input = torch.randn(32, 3, 32, 32)
output = model(dummy_input)
print(f"Output shape: {output.shape}")

total_params = 0
print("各层参数量统计：")
for name, param in model.named_parameters():
    if param.requires_grad:
        num_params = param.numel()
        total_params += num_params
        print(f"{name}: {num_params}")
print(f"模型总参数量: {total_params}") 

Output shape: torch.Size([32, 10])
各层参数量统计：
features.0.weight: 1728
features.0.bias: 64
features.3.weight: 73728
features.3.bias: 128
features.6.weight: 294912
features.6.bias: 256
features.8.weight: 589824
features.8.bias: 256
features.11.weight: 1179648
features.11.bias: 512
features.13.weight: 2359296
features.13.bias: 512
features.16.weight: 2359296
features.16.bias: 512
features.18.weight: 2359296
features.18.bias: 512
classifier.1.weight: 262144
classifier.1.bias: 512
classifier.3.weight: 5120
classifier.3.bias: 10
模型总参数量: 9488266


In [7]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
import VGG_train as mt

trainer = mt.Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    eval_step=100
)

trainer.train(20)